# Element Filtering on a Triangular Mesh
---
This notebook demonstrates how to prepare and apply an implicit filter on **mesh elements (triangles)** rather than nodes.

By default `TriangularFilter` prepares operators for node smoothing. With `filter_elements=True`, it will build the element Laplacian operator and allow filtering of element-defined data arrays.

In [ ]:
import numpy as np
import math
import matplotlib.pyplot as plt
# Mesh size
Lx = 800
Ly = Lx
# Mesh resolution
dxm = 10
dym = dxm

In [ ]:
xx = np.arange(0, Lx + 1, dxm)
yy = np.arange(0, Ly + 1, dym)

nx = len(xx)
ny = len(yy)

nodnum = np.arange(0, nx * ny)
xcoord = np.zeros((nx, ny))
ycoord = xcoord.copy()

for i in range(nx):
    ycoord[i, :] = yy
for i in range(ny):
    xcoord[:, i] = xx

from implicit_filter import make_tri
nodnum = np.reshape(nodnum, [ny, nx]).T
xcoord = np.reshape(xcoord, [nx * ny])
ycoord = np.reshape(ycoord, [nx * ny])
tri = make_tri(nodnum, nx, ny) # Creates triangulation
n2d = len(xcoord)  # The number of vertices (nodes)
e2d = len(tri[:, 1]) # Number of elements (cells)

### Create synthetic data on Elements

In [ ]:
# Generate noise directly on the elements
tt_elem = 50 * (np.random.random(e2d) - 0.5)

### Filtering
Prepare the filter with `filter_elements=True` to explicitly compute the element Laplacian operators.

In [ ]:
from implicit_filter import TriangularFilter
tri_filter = TriangularFilter()
tri_filter.prepare(n2d=n2d, e2d=e2d, tri=tri, xcoord=xcoord, ycoord=ycoord, meshtype="m", filter_elements=True, gpu=False)

In [ ]:
# Apply filter. The filter detects the size of `tt_elem` (which is e2d) and automatically uses element filtering.
tts_elem = tri_filter.compute(1, 2 * math.pi / 200., tt_elem) # Scale corresponding to approx 200 km

In [ ]:
### Visualize results
fig, ax = plt.subplots(1, 2, figsize=(16, 6))

# Plot original data
pc0 = ax[0].tripcolor(xcoord, ycoord, tri, facecolors=tt_elem, cmap='seismic')
ax[0].set_title('Original Noise on Elements')
ax[0].set_aspect('equal')
plt.colorbar(pc0, ax=ax[0])

# Plot filtered data
pc1 = ax[1].tripcolor(xcoord, ycoord, tri, facecolors=tts_elem, cmap='seismic')
ax[1].set_title(f'Filtered Elements (n=1, scale=200)')
ax[1].set_aspect('equal')
plt.colorbar(pc1, ax=ax[1])

plt.tight_layout()
plt.show()

### We can also compute the spatial spectra directly

In [ ]:
k = np.array([2 * math.pi / 200., 2 * math.pi / 100.])
spectra = tri_filter.compute_spectra_scalar(1, k, tt_elem)
print(f"Spectra: {spectra}")